# oneDNoise — `monitorVariableMeanNoiseEpochs`

Workflow notebook **tuned for the `monitorVariableMeanNoiseEpochs`
protocol** (1-D temporal noise around alternating mean intensities,
short name: `one_d_noise`). All defaults — QC thresholds, condition
keys, LN-fit knobs, population comparison axis — assume this
protocol. **For a different protocol, copy this notebook and adapt
§4 thresholds + §11 analyzer choices; do not edit oneDNoise in
place.**

## Sections at a glance

- **Setup (§1–§3)** — list protocol datafiles, pick one, build the
  `(StimBlock, ResponseBlock, AnalysisChunk)` pipeline.
- **QC + archive (§4 → §5 → §6/§9)** — automated protocol QC →
  optional click-through visual QC → per-cell PNG archive (single
  date or batch over many dates).
- **Spike-sorting QC (§7, §8)** — confirm spikes are assigned to the
  right cell.
- **Offline store (§10)** — pack QC-good cells + per-epoch
  `intensityOverFrame` traces into a single HDF5 per date.
- **Analyses (§11–§12)** — protocol-specific offline analyses
  (`retinanalysis.protocols.one_d_noise`) per date, then cross-date
  pooling.

## Workflow order (first time through)

1. Run §1 → §3 to build the pipeline for a single date.
2. Run §4 to compute `qc.csv` (auto firing-rate + silent-epoch gates).
3. Run §6 (or §9 for batch) to render the per-cell PNG archive.
4. Run §5 to tag cells `good` / `bad` interactively → `visual_qc.csv`.
5. Re-run §6 / §9: the archive now prunes to the curated set.
6. (Optional) Run §7 or §8 to sanity-check the sort itself.
7. Run §10 to write `offline.h5`, then §11 / §12 for analyses.

## Protocol summary

`monitorVariableMeanNoiseEpochs` delivers full-field Gaussian-contrast
noise: each ~2 s epoch sits at one of N mean intensities (typically
two — low / high), with noise drawn from
`current_mean × (1 + noiseStdv·randn)` and updated every `frameDwell`
monitor frames. Per-epoch logged params:

| param | what |
|---|---|
| `currentMean` | scalar, mean intensity for that epoch |
| `noiseSeed` | MATLAB MT seed (scalar) |
| `intensityOverFrame` | per-update intensity trace, length `ceil(stimTime · 60/frameDwell / 1000)` |

The new module reconstructs each epoch's per-sample stimulus from
`intensityOverFrame` (linearly interpolated to the PSTH sample rate)
and runs a cascadegraph LN fit (filter via FFT reverse-correlation,
sigmoid NL) per (cell × `currentMean`) condition.


In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


## 1. Find all experiments that ran `monitorVariableMeanNoiseEpochs`

`ra.find_available_datasets()` does a lowercase substring match against
the protocol registry, so `'monitorVariableMeanNoiseEpochs'` catches
both lab variants
(`edu.washington.riekelab.chris.protocols...` and
`edu.washington.riekelab.vyom.protocols...`). The result has **one row
per (exp_name, datafile_name)** and is already filtered to dates whose
Kilosort output is on local disk.


In [ ]:
# Protocol-registry rows for this protocol, filtered to dates with
# Kilosort output on disk. ra.find_available_datasets handles both steps
# (DJ query + os.listdir intersect) so §1 and §9 stay in sync.
exp_search = ra.find_available_datasets('monitorVariableMeanNoiseEpochs')

print(f'{len(exp_search)} usable datafile(s) found across '
      f'{exp_search.exp_name.nunique()} experiment(s).')
display(exp_search)


## 2. Pick a (date, datafile)

Edit `ENTRY_INDEX` below to a row index from §1's table. The notebook
binds `exp_name` and `datafile_name` together from that one row, so
they can't drift apart. To switch dates, change the index and re-run
from §3 down.


In [ ]:
ENTRY_INDEX = 0    # <-- EDIT ME: row index in exp_search above

# ---- Resolve the picked (exp_name, datafile_name, protocol_name) ------
_entry = exp_search.iloc[ENTRY_INDEX]
exp_name        = _entry['exp_name']
datafile_name   = _entry['datafile_name']
protocol_name   = _entry['protocol_name']
print(f'Selected entry {ENTRY_INDEX}:')
print(f'  exp_name      = {exp_name}')
print(f'  datafile_name = {datafile_name}')
print(f'  protocol_name = {protocol_name}')
print(f'  group_label   = {_entry.get("group_label", "<none>")}')

# ---- Context: every run of THIS protocol on the picked date ------------
# Some days have multiple datafiles of monitorVariableMeanNoiseEpochs
# (different NDFs, repeats, chris vs vyom variants). Show just those rows
# so it is unambiguous which one ENTRY_INDEX selected -- the picked row
# is marked with a "*".
_proto_on_day = exp_search[exp_search['exp_name'] == exp_name].copy()
_proto_on_day.insert(
    0, 'picked',
    ['*' if i == ENTRY_INDEX else '' for i in _proto_on_day.index],
)
print(f'\n{len(_proto_on_day)} monitorVariableMeanNoiseEpochs datafile(s) on '
      f'{exp_name} (picked row marked *):')
_proto_cols = [c for c in ['picked', 'exp_name', 'datafile_name',
                            'protocol_name', 'NDF', 'chunk_name',
                            'group_label']
                if c in _proto_on_day.columns]
display(_proto_on_day[_proto_cols])

# ---- Optional: full day summary (every protocol), for orientation ------
# Capped so long experiments do not blow up the notebook output. Bump
# SUMMARY_HEAD_N or call ra.get_exp_summary(exp_name) directly for more.
SUMMARY_HEAD_N = 10
_sum_df = ra.get_exp_summary(exp_name)
# Show columns that disambiguate same-day rows; keep this view narrow.
_cols = [c for c in ['data_dir', 'protocol_name', 'chunk_name',
                     'group_label', 'start_time']
         if c in _sum_df.columns]
print(f'\nfull experiment summary: {len(_sum_df)} datafiles total '
      f'(showing first {min(SUMMARY_HEAD_N, len(_sum_df))})')
display(_sum_df[_cols].head(SUMMARY_HEAD_N))

# ---- Output-folder convention for downstream cells ---------------------
# All sections that write to disk (QC, visual QC, single archive,
# sorting-QC, offline store) live under
#   <OUTPUT_DIR>/<exp_name>/<protocol_subdir>/
# Default protocol_subdir = protocol short name + datafile name
# (e.g. one_d_noise_data013) so two protocol runs on the same date
# cannot collide. Override protocol_subdir to a literal string if you
# want a custom folder name; set append_datafile_to_subdir = False to
# use the bare one_d_noise/ name.
protocol_subdir = None              # None -> auto (see append_datafile_to_subdir)
append_datafile_to_subdir = True    # appends _<datafile> to the protocol short name

# Short name used by helpers that can't derive it from a response_block
# (browse_cells_qc, load_or_build_offline, load_offline_many, ...). Must
# match the cell_plot_archive registry entry for this protocol so the
# notebook never accidentally looks under another protocol's subdir
# (e.g. eye_movement_alt_bg) just because that's the helper's default.
PROTOCOL_SHORT = 'one_d_noise'


## 3. Mirror Vision files, then build (or load) the pipeline

Building the pipeline pulls Vision files (`.ei`, `.neurons`, `.params`,
`.classification.txt`, …) for one protocol datafile and one noise
chunk — ~1 GB total. It also runs DataJoint queries and the
`cluster_match` EI alignment. Two-tier cache covers both:

### Step 3a — local file mirror (one-time bandwidth fix)

`ra.mirror_to_local_cache` copies Vision files into
`~/.cache/retinanalysis/` (override via `RA_LOCAL_CACHE_ROOT`). The
`local_cache` tier sits at the top of `find_path`'s priority list, so
any subsequent Vision read transparently uses the local copy.

### Step 3b — pipeline pkl cache (skip rebuild on restart)

`ra.create_mea_pipeline_cached` is a drop-in for
`ra.create_mea_pipeline`: pickles the built pipeline to
`<LOCAL_CACHE_ROOT>/pipelines/<key-hash>.pkl`. Subsequent calls with
the same EI-match knobs return the cached object directly — no
DataJoint queries, no `cluster_match` recompute. **Cache key includes
all EI-match knobs.**

### Pitfall

Older experiments are often sorted with `kilosort2` (not `kilosort2.5`)
— §3b auto-detects available versions. If both exist, `kilosort2.5`
wins. macOS AppleDouble dotfiles (`._kilosort*.classification.txt`) are
explicitly skipped.


In [ ]:
# Step 3a — mirror this date's Vision files to the local cache.
# Safe to re-run: files already present are skipped.
_ss_version = ra.detect_ss_version(exp_name, datafile_name)

from retinanalysis.classes.stim import MEAStimBlock
_tmp = MEAStimBlock(exp_name, datafile_name, verbose=False)
_chunk = _tmp.nearest_noise_chunk
print(f'Mirror plan:  {exp_name} / {datafile_name} (ss={_ss_version}) + chunk {_chunk}\n')

ra.compare_cache_vs_source(
    exp_name, datafile_name=datafile_name, chunk_name=_chunk,
    ss_version=_ss_version, include_sta=False,
)
print()

with ra.bandwidth_scope('Mirror (read from upstream tier)'):
    mirror_report = ra.mirror_to_local_cache(
        exp_name,
        datafile_name=datafile_name,
        chunk_name=_chunk,
        ss_version=_ss_version,
        include_sta=False,
        verbose=True,
    )
print(f'\nLocal cache root: {ra.LOCAL_CACHE_ROOT}')
print(f'  copied this run: {mirror_report["bytes_copied_total"] / 1e6:.1f} MB')
print(f'  total on disk  : {mirror_report["bytes_total"] / 1e6:.1f} MB')


In [ ]:
# Step 3b — build the pipeline (or load from local pkl cache).
ss_version = ra.detect_ss_version(exp_name, datafile_name)

# ---- USER INPUT --------------------------------------------------------
noise_chunk_name = None
typing_file_name = None

ei_corr_cutoff       = 0.6
ei_match_method      = 'all'
ei_use_isi           = False
ei_use_timecourse    = False
ei_n_removed_channels = 1

OVERWRITE_PIPELINE_CACHE = False
# ------------------------------------------------------------------------

noise_chunk_name, db_chunk_name, _chunk_warning = ra.resolve_noise_chunk(
    exp_name, datafile_name, override=noise_chunk_name,
)
if db_chunk_name is not None:
    print(f'database chunk_name for {datafile_name}: {db_chunk_name}')
print(f'using noise chunk: {noise_chunk_name}')
if _chunk_warning:
    print(f'\n*** ALERT: {_chunk_warning} ***\n')

typing_file_name = ra.pick_typing_file(
    exp_name, noise_chunk_name, ss_version, preferred=typing_file_name,
)
print(f'ss_version: {ss_version}')
print(f'typing file: {typing_file_name}')
print(f'EI match: method={ei_match_method!r} cutoff={ei_corr_cutoff} '
      f'use_isi={ei_use_isi} use_timecourse={ei_use_timecourse} '
      f'n_removed_channels={ei_n_removed_channels}')

_pkl_path = ra.pipeline_cache_path(
    exp_name, datafile_name,
    ss_version=ss_version, analysis_chunk_name=noise_chunk_name,
    typing_file=typing_file_name,
    ei_corr_cutoff=ei_corr_cutoff, ei_match_method=ei_match_method,
    ei_use_isi=ei_use_isi, ei_use_timecourse=ei_use_timecourse,
    ei_n_removed_channels=ei_n_removed_channels,
)
print(f'pkl cache: {_pkl_path}')

with ra.bandwidth_scope('Pipeline build', also_print_total=True):
    pipeline = ra.create_mea_pipeline_cached(
        exp_name,
        datafile_name,
        overwrite=OVERWRITE_PIPELINE_CACHE,
        verbose=True,
        ss_version=ss_version,
        typing_file=typing_file_name,
        analysis_chunk_name=noise_chunk_name,
        ei_corr_cutoff=ei_corr_cutoff,
        ei_match_method=ei_match_method,
        ei_use_isi=ei_use_isi,
        ei_use_timecourse=ei_use_timecourse,
        ei_n_removed_channels=ei_n_removed_channels,
    )

stim_block      = pipeline.stim
response_block  = pipeline.resp
analysis_chunk  = pipeline.analysis_chunk

print(f'\nNoise chunk used: {analysis_chunk.chunk_name}')
print(f'Cells in noise chunk: {len(analysis_chunk.cell_ids)}')
print(f'Cells in protocol datafile: {len(response_block.cell_ids)}')
print(f'Cells matched by EI: {len(pipeline.match_dict)}')


## 3c. Merge duplicate clusters before QC

Kilosort sometimes splits one physical cell across two clusters with
near-identical EIs. We merge them once, right here, so every later
section (§4 QC, §5 visual QC, §6/§9 archive, §7/§8 sorting QC, §10
offline store) sees each biological cell exactly once.

`ra.dedup_pipeline` computes pairwise EI correlations on the protocol
side, builds connected-components of clusters with `corr ≥ ei_threshold`,
picks a representative per group, and **unions** the others' spike
trains into it (with 0.5 ms refractory dedup). Type-aware: an OnP never
merges with an OffM.

Defaults match `analyze_experiment` (§6/§9), so the in-notebook
pipeline and the archived PNGs stay consistent.


In [ ]:
# §3c — Dedup the pipeline IN PLACE before any QC/analysis.
DEDUP_EI_THRESHOLD   = 0.85
DEDUP_MERGE_STRATEGY = 'union'   # 'union' merges spike trains; 'drop' keeps rep only
DEDUP_REFRACTORY_MS  = 0.5
DEDUP_SKIP_UNTYPED   = True

_n_cells_before = len(response_block.df_spike_times)
dedup_log = ra.dedup_pipeline(
    pipeline,
    ei_threshold=DEDUP_EI_THRESHOLD,
    merge_strategy=DEDUP_MERGE_STRATEGY,
    refractory_ms=DEDUP_REFRACTORY_MS,
    skip_untyped=DEDUP_SKIP_UNTYPED,
    verbose=True,
)
_n_cells_after = len(response_block.df_spike_times)
print(f'\nProtocol cells: {_n_cells_before} -> {_n_cells_after} '
      f'({_n_cells_before - _n_cells_after} merged into representatives)')

if not dedup_log['protocol'].empty:
    display(dedup_log['protocol'])


## 4. Per-cell QC inside the protocol (`monitorVariableMeanNoiseEpochs`)

This protocol is many short (~2 s) epochs, so the standard adaptive
firing-rate gate (default 1 Hz × epoch duration) is appropriate. Cells
that drop out for runs of trials are caught by the "silent epoch"
survival gate.

`protocol_qc.block_qc_metrics()` returns a per-cell metrics DataFrame;
`filter_cells_by_qc()` adds the boolean `passes` column.

**Output**: `<OUTPUT_DIR>/<exp>/<protocol_subdir>/qc.csv` — the **initial
good/bad tagging** for every cell. §5 (visual QC), §6/§9 (archives) and
§10 (offline store) all honor it.


In [ ]:
# §4 — Protocol QC.
OVERWRITE_QC = False                # True → recompute even when qc.csv exists
MIN_RATE_HZ = 1                     # firing-rate floor in spikes/s
MIN_FRAC_EPOCHS = 0.8               # fraction of epochs that must meet that rate
MIN_FRAC_NON_SILENT = 2.0 / 3.0     # cell kept iff ≥ this fraction of epochs has ≥1 spike

qc = ra.load_or_compute_protocol_qc(
    response_block, exp_name,
    protocol_subdir=protocol_subdir,
    append_datafile_to_subdir=append_datafile_to_subdir,
    datafile_name=datafile_name,
    overwrite=OVERWRITE_QC,
    min_rate_hz=MIN_RATE_HZ,
    min_frac_epochs=MIN_FRAC_EPOCHS,
    min_frac_non_silent=MIN_FRAC_NON_SILENT,
    verbose=True,
)

fails = qc[~qc.passes].sort_values('frac_non_silent_epochs')
print(f'\nFirst few failing cells ({len(fails)} total):')
display(fails[['cell_id', 'cell_type', 'n_epochs', 'mean_rate_hz',
               'frac_epochs_above_rate', 'frac_non_silent_epochs',
               'silent_run_max', 'drift_score']].head().round(2))


## 5. Visual QC (optional) — click through each cell, tag good/bad

§4 wrote an automated QC pass/fail to `qc.csv`. Use this section to
**further restrict** the archive by eyeballing each cell's raster + PSTH.

### Prerequisites

This section is **read-only on disk** — `ra.browse_cells_qc` only walks
`<OUTPUT>/<exp>/<protocol_subdir>/cells/` and appends rows to
`visual_qc.csv`. It does **not** need a pipeline, DataJoint, or any
SSD-tier read.

So you can come straight here from §2 — no §3a (mirror), §3b (pipeline
build), §3c (dedup), or §4 (QC) required — as long as:

1. The per-cell PNG archive already exists on disk from a prior §6
   (single-date) or §9 (batch) run.
2. §2 has been executed in the current kernel so `exp_name`,
   `datafile_name`, `protocol_subdir`, and `PROTOCOL_SHORT` are bound.

If the PNGs don't exist yet, the widget prints a one-line message and
returns — run §6 or §9 once to create them, then come back.

### Iterative workflow

1. First pass: **skip** this section and run §6 / §9 to build the
   initial archive (no PNGs exist yet).
2. Come back here once PNGs are on disk — `ra.browse_cells_qc(exp_name)`
   opens an ipywidgets panel that pages through cells (raster left,
   PSTH right) with `Good` / `Bad` / `Prev` / `Next` buttons. Each click
   upserts a row in `<OUTPUT>/<exp>/<protocol_subdir>/visual_qc.csv` —
   resumable across sessions.
3. Re-run §6 / §9. They auto-detect `visual_qc.csv` and restrict the
   per-cell PNG render to cells tagged `good`. `cell_match.csv` is left
   comprehensive so downstream EI joins still see the full population.

### Invariant

`visual_qc.csv` is **written only by this GUI**. `analyze_experiment`,
`save_per_cell_plots`, `save_cell_match`, and `save_protocol_qc` are all
read-only with respect to it.

Requirements: `ipywidgets` (already in the `retinanalysis` kernel).


In [ ]:
# Launch the per-cell GUI. Requires only §2 to have been run in this
# kernel (exp_name, datafile_name, protocol_subdir, PROTOCOL_SHORT) and
# the per-cell PNG archive to exist on disk from a prior §6/§9 run —
# no §3a (mirror), §3b (pipeline build), §3c (dedup), or §4 (QC) needed.
# If no PNGs exist yet, the widget prints a one-line message and returns.
ra.browse_cells_qc(
    exp_name,
    protocol=PROTOCOL_SHORT,      # set in §2 — else this helper defaults to eye_movement_alt_bg
    datafile_name=datafile_name,
    protocol_subdir=protocol_subdir,
)


## 6. Archive the picked date (single date)

`ra.analyze_experiment(exp_name, datafile_name)` writes the **full
per-cell PNG archive** for the date selected in §2. Output goes to
`<OUTPUT>/<exp>/<protocol_subdir>/`:

| file | what it is |
|---|---|
| `mosaic.png` | composite: STA mosaic + temporal-filter + ISI rows |
| `index.csv` | one row per archived cell |
| `cell_match.csv` | EI-match diagnostics per cell — kept comprehensive |
| `cells/<celltype>/cell_<id>_raster.png` | per-cell raster, condition-colored |
| `cells/<celltype>/cell_<id>_psth.png` | per-cell PSTH, condition-colored |

### Conditions used for raster + PSTH coloring

Auto-detected from the registered defaults — for `monitorVariableMeanNoiseEpochs`
that's `currentMean` (one color per mean intensity level).

### Visual-QC integration is automatic

If `visual_qc.csv` exists for this experiment (from §5), the per-cell
PNG step is restricted to cells tagged `good`. Set
`respect_visual_qc=False` to override.

### Re-archiving prunes stale PNGs

`prune_stale=True` (default): any `cells/.../cell_<id>_*.png` whose
`cell_id` is not in the kept set is deleted on re-run.


In [ ]:
# Full archive for the date picked in §2.
result = ra.analyze_experiment(
    exp_name,
    datafile_name=datafile_name,
    overwrite=True,
    fit_calibration=False,
    n_jobs=-1,
    verbose=True,
    protocol_subdir=protocol_subdir,
    append_datafile_to_subdir=append_datafile_to_subdir,
)
print(f'\nDone: {result["exp_name"]} / {result["datafile_name"]} — '
      f'QC-pass pool: {result["n_cells_passed_qc"]}/{result["n_cells_total"]}')
print(f'  output_dir: {result["output_dir"]}')


## 7. Spike-sorting QC — static PNGs (batch / remote review)

PSTH + raster confirm that spike *times* are consistent with the
stimulus; they don't tell you whether **the spikes were assigned to the
right cell** in the first place. This cell samples a few cells (QC-pass
∩ visual-QC `good`) per type and writes one PNG per cell: each row is
one full epoch (raster strip on top, 300-Hz high-pass-filtered raw
trace below, red dots at the cell's spike times).

A clean sort: red dots land on visible spike waveforms in the trace.
A merge: extra waveforms in the trace with no red dot, *or* red dots
on flat baseline.

### Output

`<OUTPUT>/<exp>/<protocol_subdir>/sorting_qc_<protocol_short>_<datafile>/cell_proto<XXXX>_noise<YYYY>_<celltype>_sorting_qc.png`


In [ ]:
# §7 — Sorting QC via raw traces, saved to disk as high-DPI PNGs.
CELL_TYPES         = ['OnP', 'OnM']
N_CELLS_PER_TYPE   = 3
N_EPOCHS           = 4
SAMPLE_STRATEGY    = 'random'
RANDOM_SEED        = None
DPI                = 250
OVERWRITE_QC_PNGS  = True

sample_df, png_paths = ra.sample_and_plot_sorting_qc(
    response_block,
    protocol_subdir=protocol_subdir if 'protocol_subdir' in dir() else None,
    append_datafile_to_subdir=(append_datafile_to_subdir
                                if 'append_datafile_to_subdir' in dir() else False),
    cell_types=CELL_TYPES,
    n_cells_per_type=N_CELLS_PER_TYPE,
    n_epochs=N_EPOCHS,
    sample_strategy=SAMPLE_STRATEGY,
    random_seed=RANDOM_SEED,
    dpi=DPI,
    overwrite=OVERWRITE_QC_PNGS,
)
print(f'\n→ wrote {len(png_paths)} PNG(s); open them with the system viewer.')


## 8. Interactive sorting-QC GUI (`ra.sorting_qc_gui`)

Notebook ipywidgets panel — same diagnostic as §7 but **one click =
one window**, on demand. Designed for remote-NAS sessions where loading
a full epoch is wasteful.

Pick cell → epoch → top-3 electrode → window (slider or FloatText) →
`Load raw trace`. Zoom / pan / appearance changes re-render from the
loaded cache (free); only sliding past the loaded window re-fetches.


In [ ]:
from IPython.display import display

display(ra.sorting_qc_gui(
    response_block,
    protocol_subdir=protocol_subdir if 'protocol_subdir' in dir() else None,
    append_datafile_to_subdir=(append_datafile_to_subdir
                                if 'append_datafile_to_subdir' in dir() else False),
))


## 9. Run the archive for one or many dates (standalone)

**Self-contained section** — run cell 1 (imports), then jump here.
`ra.analyze_experiments(dates, protocol_search=...)` packages every step
the earlier cells did manually into a single call per date.

### Defaults assume `monitorVariableMeanNoiseEpochs`

`PROTOCOL_SEARCH = "monitorVariableMeanNoiseEpochs"` substring-matches
both `chris.protocols` and `vyom.protocols` variants.

### Save toggle

`SAVE_FIGURES = True` ⇒ render and **overwrite** all PNGs.
`SAVE_FIGURES = False` ⇒ list the batch only and stop.


In [ ]:
# Section 9 is SELF-CONTAINED — you only need cell 1 (imports) to run it.

# ---- USER INPUT --------------------------------------------------------
SAVE_FIGURES = True
# ------------------------------------------------------------------------

PROTOCOL_SEARCH = 'monitorVariableMeanNoiseEpochs'

_exp_search = ra.find_available_datasets(PROTOCOL_SEARCH)
batch_dates = _exp_search['exp_name'].unique().tolist()

# Subset variants — uncomment / adapt as needed:
# batch_dates = ['20250924C', '20251008C', '20260318C']                                   # hand-pick
# batch_dates = _exp_search.query("exp_name >= '20250101C'")['exp_name'].unique().tolist() # by date

print(f'Batch run over {len(batch_dates)} dates: {batch_dates}')
print(f'SAVE_FIGURES = {SAVE_FIGURES}  '
      f'({"overwrite all PNGs" if SAVE_FIGURES else "skip archive step"})')

if not SAVE_FIGURES:
    print('SAVE_FIGURES is False — not calling ra.analyze_experiments. '
          'Set SAVE_FIGURES = True above to (re)render PNGs.')
else:
    results = ra.analyze_experiments(
        batch_dates,
        protocol_search=PROTOCOL_SEARCH,
        fit_calibration=False,
        overwrite=True,
        n_jobs=-1,
        on_error='log',
        respect_visual_qc=True,
        verbose=True,
    )
    display(ra.summarize_batch_results(results))


## 10. Offline data store (`offline.h5`) — build once, reload fast

After §5/§6 leaves a curated visual-QC set, `ra.load_or_build_offline`
packages everything an analysis needs — metadata, condition table,
per-cell spike times, smoothed PSTHs, **per-epoch `intensityOverFrame`
traces (for LN fitting)**, STA fit, EI summary — into a single HDF5 at
`<OUTPUT>/<exp>/one_d_noise/offline.h5`.

The extra `epoch_arrays/intensityOverFrame` dataset is the key
difference from the EyeMovement offline file: it's the per-update
intensity trace, length `ceil(stimTime · 60/frameDwell / 1000)`. The
analysis module reads it via `ds.epoch_array('intensityOverFrame', i)`
and linearly interpolates onto the PSTH sample grid to build the LN
stimulus.

- **First call**: builds the pipeline → runs §4 QC → intersects with
  `visual_qc.csv` (good cells only) → writes `offline.h5`. ~1–2 min/date.
- **Re-runs**: `ra.load_offline_data(exp, protocol='one_d_noise')`
  returns an `OfflineDataset` in <1 s. Pass `overwrite=True` to rebuild.
- **Cross-date**: `ra.load_offline_many(protocol='one_d_noise')` →
  `{exp_name: OfflineDataset}` for every date with `offline.h5`.


In [ ]:
# §10 — Build / load the offline store for one experiment.
EXP = exp_name

ds = ra.load_or_build_offline(
    EXP, protocol=PROTOCOL_SHORT,        # set in §2 — else default is eye_movement_alt_bg
    protocol_search='monitorVariableMeanNoiseEpochs',
    overwrite=False, verbose=True,
)
print(ds)
print('cell types:', ds.cell_types())
print('epoch_array_keys:', ds.epoch_array_keys)
display(ds.epochs.head())
display(ds.cells.head())


## 11. Offline analyses (`retinanalysis.protocols.one_d_noise`)

Each analysis takes the `OfflineDataset` from §10 and returns a dict.
LN-fit summary stats are saved next to `offline.h5` so cross-date
pooling in §12 is a single `pd.concat`.

| function | what it computes | output |
|---|---|---|
| `analyze_offline` | per-(cell-type × `currentMean`) mean PSTHs | dict (in-memory) |
| `ln_fit_per_condition` | LN model (filter + sigmoid NL) per (cell × `currentMean`) | dict + optional `ln_fits.csv` |
| `ln_fit_switching` | LN restricted to epochs that landed on `target_condition` *after* a different mean (transient-adaptation view) | dict (in-memory) |
| `ln_fit_split_phases` | LN fit per equal-time slice of each epoch — within-epoch adaptation timecourse | dict (in-memory) |
| `plot_ln_models` | filter (left) + sigmoid NL (right) for any of the above | matplotlib axes |

### LN-fit defaults (mirror MATLAB SETTINGS in
`analyzeVariableMeanNoiseMonitor.m`)

| knob | default | what |
|---|---|---|
| `filter_ms` | 1000 | length of one side of the filter |
| `frequency_cutoff_hz` | 7.5 | low-pass on the response before reverse-correlation |
| `num_bins` / `bin_type` | 50 / `'equalN'` | NL sampling |
| `correct_stim_power` | True | divide by `<S(f)·S*(f)>` to whiten |

These pull from cascadegraph (`cg.compute_filter` →
`cg.sample_nl` → `cg.SigmoidNlNode.fit_to_sample`).


In [ ]:
# §11a — Average PSTH by (cell type × currentMean). Offline = no DJ needed.
from retinanalysis.protocols import one_d_noise as odn

r = odn.analyze_offline(ds, minimum_n=3)
print(f'cell types: {r["cell_types"]}')
print(f'{len(r["conditions"])} conditions, {len(r["time_ms"])} time bins')

odn.plot_psth_by_condition(r, show_individual_cells=False)


In [ ]:
# §11b — LN model per (cell × currentMean). Filter recovered via FFT
# reverse-correlation; sigmoid NL fit to the binned generator-vs-response
# cloud. Plot filter + NL averaged across cells of one type.
ln_per_cond = odn.ln_fit_per_condition(
    ds,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    minimum_n=3,
    filter_ms=1000.0,
    frequency_cutoff_hz=7.5,
    num_bins=50,
    bin_type='equalN',
    verbose=False,
)
print(f'fit cells: '
      f'{[(ct, len(cells)) for ct, cells in ln_per_cond["fits"].items()]}')
print(f'conditions: {ln_per_cond["conditions"]}')

# Pick a cell type to visualize (first available)
_ct = ln_per_cond['cell_types'][0]
odn.plot_ln_models(ln_per_cond, cell_type=_ct, average_across_cells=True)

# Persist a per-condition summary for cross-date pooling in §12
odn.save_ln_fits(ln_per_cond, ds.exp_name)


In [ ]:
# §11c — Switching mode: LN fit restricted to epochs that landed on
# the *target* mean immediately after a different mean. With only two
# means this reduces to the single transition lowToHigh or highToLow,
# but the helper handles arbitrary mean sets.
_means = sorted(v for (v,) in ln_per_cond['conditions'])
target = _means[-1]   # the high mean by default
print(f'switching target = {target!r}; other means → {target!r}')

ln_switch = odn.ln_fit_switching(
    ds,
    target_condition=target,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    minimum_n=3,
    filter_ms=1000.0,
    frequency_cutoff_hz=7.5,
    num_bins=50,
)
print(f'transitions found: {ln_switch["transitions"]}')

_ct = ln_switch['cell_types'][0]
odn.plot_ln_models(ln_switch, cell_type=_ct, average_across_cells=True)


In [ ]:
# §11d — Phase split: slice each epoch's stim window into N equal
# chunks and refit per slice. Recovers the within-epoch adaptation
# timecourse (filter shape / NL gain as a function of time-since-step).
_means = sorted(v for (v,) in ln_per_cond['conditions'])
cond = _means[-1]    # pick the high mean to look at adaptation under it

ln_phases = odn.ln_fit_split_phases(
    ds,
    condition=cond,
    n_phases=5,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    minimum_n=3,
    filter_ms=1000.0,
    frequency_cutoff_hz=7.5,
    num_bins=50,
)
print(f'phase windows (ms): {ln_phases["phase_window_ms"]}')

_ct = ln_phases['cell_types'][0]
odn.plot_ln_models(ln_phases, cell_type=_ct, average_across_cells=True)


## 12. Cross-date aggregation

Once every experiment has been through §10–§11 (each writes
`offline.h5` and `ln_fits.csv` to its own folder), pooling across
dates is just a `concat`.

| call | returns |
|---|---|
| `ra.load_offline_many(protocol='one_d_noise')` | `{exp_name: OfflineDataset}` for every date with `offline.h5` |
| `odn.aggregate_psth_across_dates(offlines)` | pooled per-cell mean PSTHs as one `(n_cells_total, n_bins)` matrix per `(cell_type, currentMean)` — pass to `odn.plot_psth_by_condition` |
| `odn.aggregate_ln_across_dates(offlines)` | pooled filters and sigmoid params as `(n_cells_total, n_filter_pts)` / `(n_cells_total, 4)` arrays per `(cell_type, currentMean)` |
| `odn.load_ln_fits_many()` | long-format DataFrame of every saved `ln_fits.csv` |

Adding a new date to the pool: run §10–§11 for that date, then re-run
§12 unchanged — the loaders pick it up automatically.


In [ ]:
# §12 — Cross-date pooled analyses.
offlines = ra.load_offline_many(protocol=PROTOCOL_SHORT)
print(f'experiments loaded: {len(offlines)}')
for exp, ds_ in offlines.items():
    print(f'  {exp}: {len(ds_.cell_ids)} cells, types={ds_.cell_types()}')

# Pool PSTHs across dates
pooled_psth = odn.aggregate_psth_across_dates(offlines, minimum_n=5)
print(f'\npooled types (PSTH): {pooled_psth["cell_types"]} '
      f'from {pooled_psth.get("n_dates", "?")} dates')
odn.plot_psth_by_condition(pooled_psth, show_individual_cells=False)

# Pool LN fits across dates
pooled_ln = odn.aggregate_ln_across_dates(
    offlines,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    minimum_n=3,
    filter_ms=1000.0,
    frequency_cutoff_hz=7.5,
    num_bins=50,
)
print(f'\npooled types (LN): {pooled_ln["cell_types"]}')
for ct in pooled_ln['cell_types']:
    for cond, mat in pooled_ln['filters'][ct].items():
        print(f'  {ct} cond={cond}: filters {mat.shape}, '
              f'sigmoid params {pooled_ln["sigmoid_params"][ct][cond].shape}')

# Quick visualization: mean filter shape per condition, one cell type
import matplotlib.pyplot as plt
_ct = pooled_ln['cell_types'][0]
fig, ax = plt.subplots(figsize=(6, 4))
t = pooled_ln['filter_time_s']
for cond, mat in pooled_ln['filters'][_ct].items():
    normed = mat / (np.max(np.abs(mat), axis=1, keepdims=True) + 1e-12)
    mean = normed.mean(axis=0)
    sem = normed.std(axis=0) / np.sqrt(max(mat.shape[0], 1))
    label = f'currentMean={cond[0]:g}  (n={mat.shape[0]})'
    ax.plot(t, mean, lw=1.6, label=label)
    ax.fill_between(t, mean - sem, mean + sem, alpha=0.2, linewidth=0)
ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
ax.set_xlabel('time (s)')
ax.set_ylabel('filter (normalized)')
ax.set_title(f'Cross-date mean filter — {_ct}')
ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.show()

# Long-format LN summary across dates
ln_df = odn.load_ln_fits_many()
print(f'\nln_fits rows: {len(ln_df)} from '
      f'{ln_df["exp_name"].nunique() if not ln_df.empty else 0} dates')
if not ln_df.empty:
    display(ln_df.groupby(['cell_type', 'condition'])
                [['filter_peak', 'filter_peak_time_s',
                  'sigmoid_alpha', 'sigmoid_beta']]
                .agg(['median', 'count']).round(3))
